# Cấp 6 – RL Self-Play Actor-Critic & RQ3 trên Google Colab GPU T4

Phục vụ **Issue #11** và **Proposal mục 2, 4, 5** (Nhóm 31 - TTNTNC):
- **Khởi tạo:** Nạp trọng số từ mạng Cấp 5 (`MyDrive/caro/level5/best.pt`).
- **Thuật toán:** Policy Gradient có baseline (Actor-Critic) tự chơi song song theo lô (batched self-play) trên GPU T4 với nhiệt độ khám phá $\tau = 1.0$.
- **Chống quên:** Luôn giữ bản neo Cấp 5 (SL) cùng các checkpoint lịch sử gần nhất trong pool đối thủ.
- **Tự động chạy tiếp (Auto-resume):** Lưu mọi checkpoint (`best.pt`, `latest.pt`, `rounds/round_*.pt`, `pool/`) và `train_log.jsonl` vào `MyDrive/caro/level6`. Khi bị ngắt kết nối, chỉ cần chạy lại đúng ô huấn luyện để tiếp tục từ vòng dở dang.

### Hướng dẫn chạy:
1. Đảm bảo đã hoàn thành Cấp 5 và có file `/content/drive/MyDrive/caro/level5/best.pt`.
2. Vào **Runtime → Change runtime type → Chọn T4 GPU**.
3. Chạy lần lượt các ô bên dưới.

## 1. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Tải mã nguồn và cài đặt thư viện

In [ ]:
%cd /content
!test -d TTNTNC-NHOM31 || git clone https://github.com/minhhieu310800-jpg/TTNTNC-NHOM31.git
%cd /content/TTNTNC-NHOM31
!git checkout main
!git pull --ff-only
%cd do-an-caro
!pip install -q -r requirements.txt

## 3. Kiểm tra GPU T4 & Checkpoint Cấp 5

In [ ]:
import shutil
from pathlib import Path
import torch

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

DRIVE_L5 = Path('/content/drive/MyDrive/caro/level5/best.pt')
LOCAL_L5 = Path('checkpoints/level5/best.pt')
LOCAL_L5.parent.mkdir(parents=True, exist_ok=True)

if LOCAL_L5.exists():
    print(f"Đã có checkpoint Cấp 5 cục bộ: {LOCAL_L5}")
elif DRIVE_L5.exists():
    shutil.copy(DRIVE_L5, LOCAL_L5)
    print(f"Đã sao chép checkpoint Cấp 5 từ Drive về {LOCAL_L5}")
else:
    raise FileNotFoundError(
        f"Không tìm thấy checkpoint Cấp 5 tại {DRIVE_L5} hoặc {LOCAL_L5}.\n"
        "Vui lòng hoàn thành huấn luyện Cấp 5 (training/supervised-policy-value-network-colab.ipynb) trước."
    )

## 4. Huấn luyện RL Self-Play (20.000–50.000 episodes)

- Mặc định chạy 20.000 episodes (20 vòng × 1.000 ván); tăng `--episodes 50000` nếu muốn chạy thêm trên GPU.
- Khi bị ngắt giữa chừng, chỉ cần chạy lại đúng ô này để tự động tiếp tục từ `latest.pt`.

In [ ]:
OUT = '/content/drive/MyDrive/caro/level6'
!python -m training.train_rl configs/training/rl.yaml \
    --init-checkpoint checkpoints/level5/best.pt \
    --out-dir {OUT} \
    --device cuda \
    --episodes 20000 \
    --max-hours 3.5

## 5. Vẽ đường cong học tập & Báo cáo RQ3 (SL vs RL Self-Play)

In [ ]:
from IPython.display import Image, display

FIG_PATH = f"{OUT}/rq3_learning_curve.png"
REPORT_PATH = f"{OUT}/rq3_report.md"
!python -m analysis.rq3_learning_curve --train-log {OUT}/train_log.jsonl --fig {FIG_PATH} --out {REPORT_PATH}
display(Image(filename=FIG_PATH))

## 6. (Tùy chọn) Chạy giải đấu chính thức RQ3 (`configs/arena/rq3.yaml`)

Sao chép checkpoint từ Drive về `checkpoints/level6` rồi đấu đủ 200 ván/cặp trên 100 khai cuộc cân bằng (Mode B) và vẽ biểu đồ tổng hợp.

In [ ]:
import shutil
shutil.copytree(OUT, 'checkpoints/level6', dirs_exist_ok=True)
!python -m arena.tournament configs/arena/rq3.yaml
!python -m analysis.rq3_learning_curve \
    --train-log checkpoints/level6/train_log.jsonl \
    --arena-log logs/rq3_rl_vs_sl.jsonl \
    --fig {OUT}/rq3_arena_curve.png \
    --out {OUT}/rq3_arena_report.md
display(Image(filename=f"{OUT}/rq3_arena_curve.png"))

## 7. Thử nghiệm đối đầu nhanh: `RLAgent` (Cấp 6) vs `SLAgent` (Cấp 5)

In [ ]:
from game.game import CaroGame
from agents import RLAgent, SLAgent

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
rl_agent = RLAgent(checkpoint=f"{OUT}/best.pt", device=dev)
sl_agent = SLAgent(checkpoint='checkpoints/level5/best.pt', device=dev)

game = CaroGame()
while not game.is_terminal():
    if game.current_player == 1:
        action = rl_agent.select_action(game)
        tag, stats = 'RL (X)', rl_agent.last_stats
    else:
        action = sl_agent.select_action(game)
        tag, stats = 'SL (O)', sl_agent.last_stats
    r, c = divmod(action, 11)
    print(f"Nước {game.move_count + 1:2d}: {tag} -> ({r:2d}, {c:2d}) [P={stats['policy_prob']:.3f}, V={stats['value']:+.3f}]")
    game.step(action)

res = game.result()
print('Kết quả:', 'RL (X) thắng' if res == 1 else 'SL (O) thắng' if res == -1 else 'Hòa')